# 01 - Data Download

Macro data + simple trading strategy

In [ ]:
import pandas as pd
import yfinance as yf
from pandas_datareader import data as web
import matplotlib.pyplot as plt

In [ ]:
# Download data
sp500 = yf.download("^GSPC", start="2010-01-01")
vix = yf.download("^VIX", start="2010-01-01")
unrate = web.DataReader("UNRATE", "fred", "2010-01-01")

In [ ]:
# Returns
sp500["returns"] = sp500["Adj Close"].pct_change()

In [ ]:
# Merge
df = sp500[["returns"]].copy()
df["vix"] = vix["Adj Close"]
df["unrate"] = unrate["UNRATE"]
df = df.dropna()

In [ ]:
# Signal
df["vix_signal"] = (df["vix"] - df["vix"].rolling(20).mean()) / df["vix"].rolling(20).std()
df["position"] = df["vix_signal"].apply(lambda x: -1 if x > 0 else 1)

In [ ]:
# Strategy
df["strategy_returns"] = df["position"] * df["returns"]
df["cum_strategy"] = (1 + df["strategy_returns"]).cumprod()
df["cum_market"] = (1 + df["returns"]).cumprod()

In [ ]:
# Plot
plt.figure(figsize=(10,5))
plt.plot(df["cum_market"], label="Market")
plt.plot(df["cum_strategy"], label="Strategy")
plt.legend()
plt.title("Macro Strategy vs Market")
plt.show()